# Fingerprint Presentation Attack Detection (FPAD) Pipeline

This notebook allows you to interactively run the pipeline implemented in the `src/` directory. It includes feature extraction verification, dataset loading (which automatically applies **Local Binary Pattern (LBP)** feature extraction), model training, and evaluation.

> **Only one notebook to run:** `run_pipeline.ipynb` is the canonical entry point for the entire pipeline. Open it, click **Run All**, and the full experiment executes from start to finish.

In [ ]:
import sys
import os
import numpy as np
from IPython.display import Image, display

# Add the src directory to the path so we can import our modules
sys.path.append(os.path.abspath('src'))

from main import load_fvc2000_data, load_socofing_data, run_pipeline

## 1. Feature Extraction Verification

Before running the full machine learning pipeline, we verify that the **Local Binary Pattern (LBP)** feature extraction is working exactly as mathematically specified (640-dimensional vector, correct blocks and bins).

This runs the standalone verification script and displays some of the generated diagnostic visuals.

In [ ]:
# Run the rigorous verification script
!python src/lbp_verify.py

# Display some of the diagnostic visualizations generated by the script
print("\n--- Feature Extraction Visualizations ---")
display(Image(filename="outputs/visualizations/sample_original_vs_lbp.png"))
display(Image(filename="outputs/visualizations/sample_block_histogram.png"))

## 2. FVC2000 Dataset Experiment

### Dataset Description & Expected Results

| Folder | Class | Sensor | # Images (Set A / Test) |
|--------|-------|--------|------------------------|
| Db1_a  | Genuine (1) | KeyTronic Optical   | 800 |
| Db2_a  | Genuine (1) | ST TouchChip Capacitive | 800 |
| Db3_a  | Genuine (1) | Identicator Optical | 800 |
| Db4_a  | Fake (0)    | **SFinGe Synthetic Generator** | 800 |

**Why results are near-perfect on FVC2000 (ROC-AUC ≈ 1.0) — this is expected and NOT a bug:**

> The FVC2000 DB4 is **100% synthetic**, generated by a mathematical software tool called **SFinGe**. Databases 1–3 are captured on real hardware sensors and contain true sensor noise, sweat pores, and optical distortions. Local Binary Pattern (LBP) is a micro-texture descriptor — it is exquisitely sensitive to surface noise. The synthetic DB4 ridges lack all real sensor artefacts, making them trivially distinguishable from real sensor images in LBP-space. This is a well-documented finding in the literature. It does **not** indicate data leakage or overfitting.

> **Data leakage check:** Training uses Set B (_b folders, fingers 101–110) and testing uses Set A (_a folders, fingers 1–100). These are completely disjoint — **zero image overlap** has been verified programmatically.

### Step 2a: LBP Feature Extraction & Dataset Splitting

In [ ]:
# Load FVC2000 Data and Extract LBP Features
X_train_fvc, X_test_fvc, y_train_fvc, y_test_fvc = load_fvc2000_data()

print("\n=== FVC2000 Dataset Breakdown ===")
print(f"Feature matrix dimensionality (LBP): {X_train_fvc.shape[1]}")
print(f"Training Samples  : {X_train_fvc.shape[0]} (Genuine: {(y_train_fvc==1).sum()} | Fake: {(y_train_fvc==0).sum()}) [Balanced by undersampling]")
print(f"Testing Samples   : {X_test_fvc.shape[0]} (Genuine: {(y_test_fvc==1).sum()} | Fake: {(y_test_fvc==0).sum()}) [Real-world imbalance]")
print()
print("NOTE: Near-perfect AUC (~1.0) on FVC2000 is EXPECTED.")
print("DB4 is synthetic (SFinGe) — LBP trivially distinguishes it from real sensor images.")
print("This is a known dataset characteristic, NOT a bug or data leakage.")

### Step 2b: Training & Evaluation

In [ ]:
# Run Pipeline (This will tune hyperparameters using GridSearchCV, print metrics, and save plots)
run_pipeline("FVC2000", X_train_fvc, X_test_fvc, y_train_fvc, y_test_fvc)

In [ ]:
# Display the generated FVC2000 evaluation plots
print("\n--- FVC2000 ROC Curves ---")
display(Image(filename="results/FVC2000_SVM_roc.png"))
display(Image(filename="results/FVC2000_KNN_roc.png"))

print("\n--- FVC2000 Confusion Matrices ---")
display(Image(filename="results/FVC2000_SVM_confusion_matrix.png"))
display(Image(filename="results/FVC2000_KNN_confusion_matrix.png"))

## 3. SOCOFing Dataset Experiment

### Dataset Description & Why This Is the Real Challenge

| Folder | Class | Description | # Images |
|--------|-------|-------------|----------|
| Real/  | Genuine (1) | Real sensor captures | 6,000 |
| Altered-Easy/   | Fake (0) | Digitally altered — Obliteration | ~16,000 |
| Altered-Medium/ | Fake (0) | Digitally altered — Central Rotation | ~16,000 |
| Altered-Hard/   | Fake (0) | Digitally altered — Z-Cut | ~17,000 |

**Why SOCOFing is the scientifically meaningful benchmark:**

> SOCOFing fakes are **real fingerprints that have been digitally altered**, not synthetic images. Both the genuine and altered classes originate from the same sensor, so they share the same noise profile and texture statistics. The LBP descriptor must now detect subtle structural modifications rather than gross synthetic vs. real differences. This produces **realistic, non-trivial ROC-AUC scores** (typically 0.65–0.85 depending on difficulty tier).

> **Class imbalance strategy:** The pipeline applies majority-class undersampling to the **training set only** (keeping it balanced for fair learning), while the **test set retains its real-world imbalanced distribution** for honest evaluation.

### Step 3a: LBP Feature Extraction & Stratified Splitting

> **Note:** We use `sample_fraction=0.05` (5%) here for a fast dry run (~2,600 images). Set it to `1.0` to run on the entire ~55,000 image dataset.

In [ ]:
# Load SOCOFing Data (5% sample for speed — set to 1.0 for full dataset)
X_train_soco, X_test_soco, y_train_soco, y_test_soco = load_socofing_data(sample_fraction=0.05)

print("\n=== SOCOFing Dataset Breakdown ===")
print(f"Feature matrix dimensionality (LBP): {X_train_soco.shape[1]}")
print(f"Training Samples  : {X_train_soco.shape[0]} (Genuine: {(y_train_soco==1).sum()} | Fake: {(y_train_soco==0).sum()}) [Balanced by undersampling]")
print(f"Testing Samples   : {X_test_soco.shape[0]} (Genuine: {(y_test_soco==1).sum()} | Fake: {(y_test_soco==0).sum()}) [Imbalanced — Real-World]")
print()
print("NOTE: Realistic (non-trivial) AUC expected on SOCOFing.")
print("Fakes are real sensor images with digital alterations — much harder for LBP to distinguish.")

### Step 3b: Training & Evaluation

In [ ]:
# Run Pipeline
run_pipeline("SOCOFing", X_train_soco, X_test_soco, y_train_soco, y_test_soco)

In [ ]:
# Display the generated SOCOFing evaluation plots
print("\n--- SOCOFing ROC Curves ---")
display(Image(filename="results/SOCOFing_SVM_roc.png"))
display(Image(filename="results/SOCOFing_KNN_roc.png"))

print("\n--- SOCOFing Confusion Matrices ---")
display(Image(filename="results/SOCOFing_SVM_confusion_matrix.png"))
display(Image(filename="results/SOCOFing_KNN_confusion_matrix.png"))